# Phase 1: Hybrid Legal Language Model


## 1. Check GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

## 3. Hugging Face login


In [ ]:
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
login(token=HF_TOKEN)


## 4. Load Tokenizer
Reuse the byte-level BPE implementation. The first four special tokens stay in their original order; `<mask>` is added as a fifth special token for corruption. The tokenizer is trained only on the training-document split. Do not reuse checkpoints with a different vocabulary mapping.


In [ ]:
from collections import defaultdict, Counter
import heapq
SPECIALS = ['<pad>', '<unk>', '<s>', '</s>', '<mask>']

class BytelevelBPE:

    def __init__(self, vocab_size=5000):
        self.vocab_size = vocab_size
        self.specials = SPECIALS.copy()
        self.byte_vocab = [str(i) for i in range(256)]
        self.merges = []
        self.merge_ranks = {}
        self.vocab = []
        self.stoi = {}
        self.itos = {}
        self.symbol_to_bytes = {}

    def word_to_symbols(self, word):
        byte_seq = word.encode('utf-8')
        return [str(b) for b in byte_seq] + ['</w>']

    def build_word_freqs(self, corpus):
        word_counter = Counter()
        print('Building word frequencies...')
        for text in corpus:
            if not isinstance(text, str):
                continue
            word_counter.update(text.split())
        print(f'Unique whitespace-separated words: {len(word_counter):,}')
        word_freqs = {}
        for word, freq in word_counter.items():
            symbols = tuple(self.word_to_symbols(word))
            word_freqs[symbols] = word_freqs.get(symbols, 0) + freq
        return word_freqs

    @staticmethod
    def get_pairs(symbols):
        return {(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)}

    @staticmethod
    def pair_counts_in_word(symbols):
        counts = Counter()
        for i in range(len(symbols) - 1):
            counts[symbols[i], symbols[i + 1]] += 1
        return counts

    @staticmethod
    def merge_symbols(symbols, pair, merged_symbol):
        new_symbols = []
        i = 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == pair[0] and (symbols[i + 1] == pair[1]):
                new_symbols.append(merged_symbol)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        return tuple(new_symbols)

    def train(self, corpus):
        word_freqs = self.build_word_freqs(corpus)
        vocab = self.specials + self.byte_vocab + ['</w>']
        seen = set(vocab)
        for b in range(256):
            self.symbol_to_bytes[str(b)] = bytes([b])
        self.symbol_to_bytes['</w>'] = b''
        words = []
        frequencies = []
        for symbols, freq in word_freqs.items():
            words.append(symbols)
            frequencies.append(freq)
        del word_freqs
        pair_counts = defaultdict(int)
        pair_to_words = defaultdict(set)
        print('Building initial pair statistics...')
        for word_id, symbols in enumerate(words):
            freq = frequencies[word_id]
            local_counts = self.pair_counts_in_word(symbols)
            for pair, count in local_counts.items():
                pair_counts[pair] += count * freq
                pair_to_words[pair].add(word_id)
        print(f'Initial unique pairs: {len(pair_counts):,}')
        heap = []
        for pair, count in pair_counts.items():
            heapq.heappush(heap, (-count, pair))
        target_merges = max(0, self.vocab_size - len(vocab))
        print(f'Target merges: {target_merges:,}')
        while len(vocab) < self.vocab_size and heap:
            while heap:
                neg_count, pair = heapq.heappop(heap)
                heap_count = -neg_count
                current_count = pair_counts.get(pair, 0)
                if current_count == heap_count and current_count > 0:
                    break
            else:
                break
            max_freq = pair_counts.get(pair, 0)
            if max_freq <= 1:
                break
            merged_symbol = f'<bpe_{len(self.merges)}>'
            left_bytes = self.symbol_to_bytes[pair[0]]
            right_bytes = self.symbol_to_bytes[pair[1]]
            self.symbol_to_bytes[merged_symbol] = left_bytes + right_bytes
            self.merges.append(pair)
            self.merge_ranks[pair] = (len(self.merges) - 1, merged_symbol)
            if merged_symbol not in seen:
                seen.add(merged_symbol)
                vocab.append(merged_symbol)
            affected_word_ids = list(pair_to_words.get(pair, set()))
            changed_pairs = set()
            for word_id in affected_word_ids:
                old_symbols = words[word_id]
                freq = frequencies[word_id]
                if pair not in self.get_pairs(old_symbols):
                    continue
                old_counts = self.pair_counts_in_word(old_symbols)
                new_symbols = self.merge_symbols(old_symbols, pair, merged_symbol)
                if new_symbols == old_symbols:
                    continue
                new_counts = self.pair_counts_in_word(new_symbols)
                words[word_id] = new_symbols
                all_pairs = set(old_counts) | set(new_counts)
                for p in all_pairs:
                    old_n = old_counts.get(p, 0)
                    new_n = new_counts.get(p, 0)
                    delta = (new_n - old_n) * freq
                    if delta != 0:
                        pair_counts[p] += delta
                        changed_pairs.add(p)
                    if new_n > 0:
                        pair_to_words[p].add(word_id)
                    else:
                        pair_to_words[p].discard(word_id)
            pair_to_words[pair].clear()
            for p in changed_pairs:
                count = pair_counts.get(p, 0)
                if count > 0:
                    heapq.heappush(heap, (-count, p))
        self.vocab = vocab
        self.vocab_size = len(vocab)
        self.stoi = {symbol: i for i, symbol in enumerate(vocab)}
        self.itos = {i: symbol for symbol, i in self.stoi.items()}
        print('\nTokenizer training complete.')
        print('Final vocabulary size:', self.vocab_size)
        print('Number of merges:', len(self.merges))

    def apply_bpe(self, word):
        symbols = self.word_to_symbols(word)
        while len(symbols) > 1:
            best_pair = None
            best_rank = float('inf')
            best_merged_symbol = None
            for i in range(len(symbols) - 1):
                pair = (symbols[i], symbols[i + 1])
                merge_info = self.merge_ranks.get(pair)
                if merge_info is not None:
                    rank, merged_symbol = merge_info
                    if rank < best_rank:
                        best_rank = rank
                        best_pair = pair
                        best_merged_symbol = merged_symbol
            if best_pair is None:
                break
            symbols = list(self.merge_symbols(tuple(symbols), best_pair, best_merged_symbol))
        return symbols

    def encode(self, text, add_bos=False, add_eos=False):
        ids = []
        if add_bos:
            ids.append(self.stoi['<s>'])
        for word in text.split():
            symbols = self.apply_bpe(word)
            for symbol in symbols:
                ids.append(self.stoi.get(symbol, self.stoi['<unk>']))
        if add_eos:
            ids.append(self.stoi['</s>'])
        return ids

    def decode(self, ids):
        words = []
        current_bytes = bytearray()
        for token_id in ids:
            symbol = self.itos.get(token_id)
            if symbol is None:
                continue
            if symbol in self.specials:
                continue
            ends_word = self._symbol_ends_word(symbol)
            byte_value = self.symbol_to_bytes.get(symbol, b'')
            current_bytes.extend(byte_value)
            if ends_word:
                try:
                    word = bytes(current_bytes).decode('utf-8')
                    words.append(word)
                except UnicodeDecodeError:
                    words.append('<unk>')
                current_bytes = bytearray()
        if current_bytes:
            try:
                words.append(bytes(current_bytes).decode('utf-8'))
            except UnicodeDecodeError:
                words.append('<unk>')
        return ' '.join(words)

    def _symbol_ends_word(self, symbol):
        if symbol == '</w>':
            return True
        for pair, (_, merged) in self.merge_ranks.items():
            if merged == symbol:
                return self._symbol_ends_word(pair[1])
        return False

    def get_vocab_size(self):
        return len(self.vocab)


## 9. Define The Model
Each block has gated multi-head attention and a GEGLU feed-forward layer, with residual connections. The same weights process both objectives. `causal=False` gives bidirectional encoder attention. `causal=True` gives left-to-right attention. KV caching is allowed only in causal evaluation, because adding a token in bidirectional mode would change earlier representations.


In [ ]:
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class HybridConfig:
    vocab_size: int
    pad_id: int
    mask_id: int
    d_model: int = 512
    n_layers: int = 12
    n_heads: int = 8
    d_ff: int = 1664
    max_seq_len: int = 512
    dropout: float = 0.1


class GatedAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_heads = cfg.n_heads
        self.head_dim = cfg.d_model // cfg.n_heads
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=False)
        self.gate = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.out = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.output_norm = nn.LayerNorm(cfg.d_model, elementwise_affine=False)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x, key_valid, causal=True, past=None, use_cache=False):
        b, t, d = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        past_len = 0 if past is None else past[0].size(2)
        if past is not None:
            k = torch.cat((past[0], k), dim=2)
            v = torch.cat((past[1], v), dim=2)

        allowed = key_valid[:, None, None, :]
        if causal:
            query_positions = torch.arange(past_len, past_len + t, device=x.device)
            key_positions = torch.arange(k.size(2), device=x.device)
            allowed = allowed & (key_positions[None, :] <= query_positions[:, None])[None, None]

        # A fully padded query row gets a harmless fallback key to avoid NaNs.
        first_key = torch.arange(k.size(2), device=x.device).eq(0)
        allowed = allowed | (~allowed.any(dim=-1, keepdim=True) & first_key)
        # Explicit mask also handles offset queries when a KV cache is present.
        context = F.scaled_dot_product_attention(
            q, k, v, attn_mask=allowed,
            dropout_p=self.dropout.p if self.training else 0.0,
            is_causal=False,
        )
        context = context.transpose(1, 2).contiguous().view(b, t, d)
        # Paper Section 2.2: GEGLU gate and non-affine output LayerNorm.
        context = context * F.gelu(self.gate(x))
        output = self.out(self.output_norm(context))
        return output, (k, v) if use_cache else None


In [ ]:
class GatedFeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.value = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.output_norm = nn.LayerNorm(cfg.d_ff, elementwise_affine=False)
        self.out = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)

    def forward(self, x):
        hidden = self.value(x) * F.gelu(self.gate(x))
        return self.out(self.output_norm(hidden))


class HybridBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = nn.LayerNorm(cfg.d_model, elementwise_affine=False)
        self.norm2 = nn.LayerNorm(cfg.d_model, elementwise_affine=False)
        self.attn = GatedAttention(cfg)
        self.ffn = GatedFeedForward(cfg)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x, valid, causal, past=None, use_cache=False):
        attention, kv = self.attn(self.norm1(x), valid, causal, past, use_cache)
        x = x + self.dropout(attention)
        x = x + self.dropout(self.ffn(self.norm2(x)))
        return x, kv


class GPTBertLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=cfg.pad_id)
        self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model)
        self.embedding_norm = nn.LayerNorm(cfg.d_model, elementwise_affine=False)
        self.dropout = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([HybridBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = nn.LayerNorm(cfg.d_model, elementwise_affine=False)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.apply(self._initialize)
        self.lm_head.weight = self.tok_emb.weight
        # Smaller residual projections keep a deep stack stable at initialization.
        for block in self.blocks:
            nn.init.normal_(block.attn.out.weight, std=0.02 / math.sqrt(2 * cfg.n_layers))
            nn.init.normal_(block.ffn.out.weight, std=0.02 / math.sqrt(2 * cfg.n_layers))
        with torch.no_grad():
            self.tok_emb.weight[cfg.pad_id].zero_()

    def _initialize(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, std=0.02)

    def forward(self, ids, causal=True, cache=None, use_cache=False, last_only=False):
        if (cache is not None or use_cache) and (not causal or self.training):
            raise ValueError('KV caching is only used in causal evaluation mode.')
        offset = 0 if cache is None else cache['length']
        if offset + ids.size(1) > self.cfg.max_seq_len:
            raise ValueError('Context is full: rebuild from the most recent tokens.')
        positions = torch.arange(offset, offset + ids.size(1), device=ids.device)
        x = self.dropout(self.embedding_norm(self.tok_emb(ids) + self.pos_emb(positions)))
        valid = ids.ne(self.cfg.pad_id)
        if cache is not None:
            valid = torch.cat((cache['valid'], valid), dim=1)
        updated = []
        for i, block in enumerate(self.blocks):
            past = None if cache is None else cache['kv'][i]
            x, kv = block(x, valid, causal, past, use_cache)
            if use_cache:
                updated.append(kv)
        logits = self.lm_head(self.final_norm(x[:, -1:] if last_only else x))
        if use_cache:
            return logits, dict(kv=updated, valid=valid, length=offset + ids.size(1))
        return logits


# **Phase 5**

In [ ]:
import os
import json
import random
import zipfile
from pathlib import Path
from dataclasses import asdict

import numpy as np
import torch
import torch.nn as nn

from huggingface_hub import hf_hub_download
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint as activation_checkpoint
from torch.nn.utils.rnn import (
    pad_sequence,
    pack_padded_sequence,
    pad_packed_sequence,
)
from sklearn.metrics import f1_score, accuracy_score

MODEL_DIR = Path(
    "/kaggle/input/datasets/sruthigms/model3-for-phase5-files"
)

OUTPUT_DIR = Path("/kaggle/working/phase5_hybrid_lora")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42

CHUNK_LENGTH = 512
MAX_CHUNKS = 16
DOC_BATCH_SIZE = 8
CHUNK_BATCH_SIZE = 32

LORA_LAST_N = 4
RANK = 16
ALPHA = 32

ATTENTION_TARGETS = ["qkv", "gate", "out"]
FFN_TARGETS = ["value", "gate", "out"]

GRU_HIDDEN = 256
HEAD_DROPOUT = 0.2

LORA_LR = 1e-4
HEAD_LR = 1e-4

EPOCHS = 6
PATIENCE = 3
USE_ACTIVATION_CHECKPOINTING = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

use_amp = device.type == "cuda"

amp_dtype = (
    torch.bfloat16
    if use_amp and torch.cuda.is_bf16_supported()
    else torch.float16
)

In [ ]:
saved = torch.load(
    MODEL_DIR / "model.pt",
    map_location="cpu",
    weights_only=True,
)

cfg = HybridConfig(**saved["model_config"])

phase1_model = GPTBertLM(cfg)
phase1_model.load_state_dict(saved["model_state_dict"])

del saved

with open(MODEL_DIR / "tokenizer.json", encoding="utf-8") as f:
    state = json.load(f)

tokenizer = BytelevelBPE(vocab_size=state["vocab_size"])
tokenizer.specials = state["specials"]
tokenizer.vocab = state["vocab"]
tokenizer.vocab_size = len(tokenizer.vocab)

tokenizer.stoi = {
    symbol: i for i, symbol in enumerate(tokenizer.vocab)
}
tokenizer.itos = {
    i: symbol for symbol, i in tokenizer.stoi.items()
}
tokenizer.merges = [
    tuple(pair) for pair in state["merges"]
]
tokenizer.merge_ranks = {
    tuple(record["pair"]): (
        record["rank"],
        record["merged_symbol"],
    )
    for record in state["merge_ranks"]
}
tokenizer.symbol_to_bytes = {
    symbol: bytes(values)
    for symbol, values in state["symbol_to_bytes"].items()
}

del state

PAD_ID = tokenizer.stoi["<pad>"]
UNK_ID = tokenizer.stoi["<unk>"]
BOS_ID = tokenizer.stoi["<s>"]
EOS_ID = tokenizer.stoi["</s>"]

CHUNK_LENGTH = min(CHUNK_LENGTH, cfg.max_seq_len)

In [ ]:
DATA_DIR = Path("/kaggle/working/capstone_data")

for filename in [
    "Phase_5_LSI/Train/train.json",
    "Phase_5_LSI/Train/dev.json",
    "Phase_5_LSI/Test/test.json",
]:
    hf_hub_download(
        repo_id="Exploration-Lab/CS-781-Capstone",
        repo_type="dataset",
        filename=filename,
        local_dir=str(DATA_DIR),
        token=globals().get("HF_TOKEN") or os.getenv("HF_TOKEN"),
    )

LSI_DIR = DATA_DIR / "Phase_5_LSI"


def read_cases(path, labeled=True):
    with open(path, encoding="utf-8") as f:
        records = json.load(f)

    cases = []
    seen = {}

    for record in records:
        labels = (
            sorted({int(label) for label in record["labels"]})
            if labeled else []
        )

        case = {
            "id": record["id"],
            "text": record["text"],
            "labels": labels,
        }

        if labeled:
            key = str(case["id"])
            value = (case["text"], tuple(labels))

            if key in seen:
                if seen[key] != value:
                    raise ValueError(f"Conflicting case ID: {key}")
                continue

            seen[key] = value

        cases.append(case)

    return cases


train_cases = read_cases(LSI_DIR / "Train/train.json")
dev_cases = read_cases(LSI_DIR / "Train/dev.json")

In [ ]:
statute_ids = sorted({
    label
    for case in train_cases + dev_cases
    for label in case["labels"]
})

label2index = {
    label: index for index, label in enumerate(statute_ids)
}

NUM_LABELS = len(statute_ids)


def make_target(labels):
    target = torch.zeros(NUM_LABELS, dtype=torch.float32)

    for label in labels:
        target[label2index[label]] = 1.0

    return target

In [ ]:
class StatuteDataset(Dataset):
    def __init__(self, cases):
        self.cases = cases
        self.chunks = []
        self.targets = [
            make_target(case["labels"]) for case in cases
        ]

        capacity = CHUNK_LENGTH - 2

        for i, case in enumerate(cases):
            ids = tokenizer.encode(case["text"]) or [UNK_ID]
            starts = list(range(0, len(ids), capacity))

            if len(starts) > MAX_CHUNKS:
                selected = np.linspace(
                    0, len(starts) - 1, MAX_CHUNKS, dtype=int
                )
                starts = [starts[j] for j in selected]

            self.chunks.append([
                np.array(
                    [BOS_ID]
                    + ids[start:start + capacity]
                    + [EOS_ID],
                    dtype=np.int32,
                )
                for start in starts
            ])

            if (i + 1) % 1000 == 0:
                print(f"Tokenized {i + 1}/{len(cases)}")

    def __len__(self):
        return len(self.cases)

    def __getitem__(self, index):
        return {
            "id": self.cases[index]["id"],
            "labels": self.targets[index],
            "chunks": [
                torch.tensor(chunk, dtype=torch.long)
                for chunk in self.chunks[index]
            ],
        }


def collate_cases(examples):
    chunks = [
        chunk
        for example in examples
        for chunk in example["chunks"]
    ]

    return {
        "input_ids": pad_sequence(
            chunks,
            batch_first=True,
            padding_value=PAD_ID,
        ),
        "lengths": torch.tensor([
            len(chunk) for chunk in chunks
        ]),
        "chunk_counts": torch.tensor([
            len(example["chunks"]) for example in examples
        ]),
        "labels": torch.stack([
            example["labels"] for example in examples
        ]),
        "ids": [example["id"] for example in examples],
    }


def make_loader(dataset, shuffle=False):
    return DataLoader(
        dataset,
        batch_size=DOC_BATCH_SIZE,
        shuffle=shuffle,
        collate_fn=collate_cases,
        pin_memory=use_amp,
    )


def move_batch(batch):
    for key in ["input_ids", "lengths", "labels"]:
        batch[key] = batch[key].to(
            device, non_blocking=True
        )

    # Keep chunk_counts on CPU for pack_padded_sequence.
    return batch


train_dataset = StatuteDataset(train_cases)
dev_dataset = StatuteDataset(dev_cases)

train_loader = make_loader(train_dataset, shuffle=True)
dev_loader = make_loader(dev_dataset)

In [ ]:
class LoRA(nn.Module):
    def __init__(self, base, rank, alpha):
        super().__init__()

        self.base = base
        self.base.requires_grad_(False)
        self.scale = alpha / rank

        self.A = nn.Linear(
            base.in_features,
            rank,
            bias=False,
            device=base.weight.device,
            dtype=base.weight.dtype,
        )

        self.B = nn.Linear(
            rank,
            base.out_features,
            bias=False,
            device=base.weight.device,
            dtype=base.weight.dtype,
        )

        nn.init.normal_(self.A.weight, std=0.02)
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        return self.base(x) + self.scale * self.B(self.A(x))


class HybridLSI(nn.Module):
    def __init__(self, base, num_labels):
        super().__init__()

        base.requires_grad_(False)

        self.tok_emb = base.tok_emb
        self.pos_emb = base.pos_emb
        self.embedding_norm = base.embedding_norm
        self.embedding_dropout = base.dropout
        self.blocks = base.blocks
        self.final_norm = base.final_norm

        self.adapter_start = max(
            0, len(self.blocks) - LORA_LAST_N
        )

        self.chunk_batch_size = CHUNK_BATCH_SIZE
        self.use_checkpointing = USE_ACTIVATION_CHECKPOINTING

        for block in self.blocks[self.adapter_start:]:
            for name in ATTENTION_TARGETS:
                layer = getattr(block.attn, name)

                setattr(
                    block.attn,
                    name,
                    LoRA(layer, RANK, ALPHA),
                )

            for name in FFN_TARGETS:
                layer = getattr(block.ffn, name)

                setattr(
                    block.ffn,
                    name,
                    LoRA(layer, RANK, ALPHA),
                )

        self.gru = nn.GRU(
            input_size=base.cfg.d_model,
            hidden_size=GRU_HIDDEN,
            batch_first=True,
            bidirectional=True,
        )

        self.attention = nn.Sequential(
            nn.Linear(2 * GRU_HIDDEN, 128),
            nn.Tanh(),
            nn.Linear(128, 1, bias=False),
        )

        self.dropout = nn.Dropout(HEAD_DROPOUT)
        self.classifier = nn.Linear(
            2 * GRU_HIDDEN, num_labels
        )

    def train(self, mode=True):
        super().train(mode)

        self.embedding_dropout.eval()

        for block in self.blocks[:self.adapter_start]:
            block.eval()

        return self

    def encode_chunks(self, input_ids, lengths):
        positions = torch.arange(
            input_ids.size(1),
            device=input_ids.device,
        )

        valid = positions[None, :] < lengths[:, None]

        x = self.embedding_norm(
            self.tok_emb(input_ids)
            + self.pos_emb(positions)
        )
        x = self.embedding_dropout(x)

        for i, block in enumerate(self.blocks):
            if (
                self.training
                and self.use_checkpointing
                and i >= self.adapter_start
            ):
                x, _ = activation_checkpoint(
                    block,
                    x,
                    valid,
                    False,  # Bidirectional attention.
                    use_reentrant=False,
                )
            else:
                x, _ = block(x, valid, False)

        x = self.final_norm(x).float()

        # Exclude BOS, EOS, and padding from mean pooling.
        text_mask = (
            (positions[None, :] >= 1)
            & (positions[None, :] < lengths[:, None] - 1)
        ).unsqueeze(-1).float()

        return (
            (x * text_mask).sum(1)
            / text_mask.sum(1).clamp_min(1)
        )

    def forward(self, input_ids, lengths, chunk_counts):
        vectors = []

        for start in range(
            0, len(input_ids), self.chunk_batch_size
        ):
            end = start + self.chunk_batch_size

            vectors.append(
                self.encode_chunks(
                    input_ids[start:end],
                    lengths[start:end],
                )
            )

        vectors = torch.cat(vectors)

        sequences = torch.split(
            vectors, chunk_counts.tolist()
        )
        x = pad_sequence(sequences, batch_first=True)

        with torch.autocast(
            device_type=x.device.type,
            enabled=False,
        ):
            x = self.dropout(x.float())

            packed = pack_padded_sequence(
                x,
                chunk_counts,
                batch_first=True,
                enforce_sorted=False,
            )

            packed, _ = self.gru(packed)

            x, _ = pad_packed_sequence(
                packed, batch_first=True
            )

            positions = torch.arange(
                x.size(1), device=x.device
            )[None, :]

            valid = (
                positions
                < chunk_counts.to(x.device)[:, None]
            )

            scores = self.attention(x).squeeze(-1)
            scores = scores.masked_fill(
                ~valid, float("-inf")
            )

            weights = torch.softmax(scores, dim=1)

            document_vectors = (
                x * weights.unsqueeze(-1)
            ).sum(1)

            return self.classifier(
                self.dropout(document_vectors)
            )


torch.manual_seed(SEED)

lsi_model = HybridLSI(
    phase1_model, NUM_LABELS
).to(device)

del phase1_model

In [ ]:
lora_parameters = []
head_parameters = []

for name, parameter in lsi_model.named_parameters():
    if not parameter.requires_grad:
        continue

    if name.startswith("blocks."):
        lora_parameters.append(parameter)
    else:
        head_parameters.append(parameter)

optimizer = torch.optim.AdamW(
    [
        {
            "params": lora_parameters,
            "lr": LORA_LR,
        },
        {
            "params": head_parameters,
            "lr": HEAD_LR,
        },
    ],
    weight_decay=0.01,
)

criterion = nn.BCEWithLogitsLoss()

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp and amp_dtype == torch.float16,
)

In [ ]:
def get_logits(model, batch):
    return model(
        batch["input_ids"],
        batch["lengths"],
        batch["chunk_counts"],
    ).float()


@torch.inference_mode()
def collect_probabilities(model, loader):
    model.eval()

    probabilities = []
    targets = []

    for batch in loader:
        batch = move_batch(batch)

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=use_amp,
        ):
            logits = get_logits(model, batch)

        probabilities.append(
            torch.sigmoid(logits).cpu()
        )
        targets.append(batch["labels"].cpu())

    return (
        torch.cat(probabilities).numpy(),
        torch.cat(targets).numpy().astype(np.int32),
    )


def choose_threshold(probabilities, targets):
    results = []

    for threshold in np.round(
        np.arange(0.01, 1.00, 0.01), 2
    ):
        predictions = probabilities >= threshold

        score = f1_score(
            targets,
            predictions,
            average="macro",
            zero_division=0,
        )

        results.append(
            (float(threshold), float(score))
        )

    return min(
        results,
        key=lambda row: (-row[1], abs(row[0] - 0.5)),
    )

In [ ]:
model_settings = {
    "phase1_config": asdict(cfg),
    "adaptation": "lora",
    "lora_last_n": LORA_LAST_N,
    "rank": RANK,
    "alpha": ALPHA,
    "attention_targets": ATTENTION_TARGETS.copy(),
    "ffn_targets": FFN_TARGETS.copy(),
    "causal": False,
    "gru_hidden": GRU_HIDDEN,
    "head_dropout": HEAD_DROPOUT,
    "chunk_length": CHUNK_LENGTH,
    "max_chunks": MAX_CHUNKS,
}

best_f1 = -1.0
bad_epochs = 0

for epoch in range(1, EPOCHS + 1):
    lsi_model.train()

    total_loss = 0.0
    total_cases = 0

    for step, batch in enumerate(train_loader, start=1):
        batch = move_batch(batch)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=use_amp,
        ):
            logits = get_logits(lsi_model, batch)
            loss = criterion(logits, batch["labels"])

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        nn.utils.clip_grad_norm_(
            lsi_model.parameters(), 1.0
        )

        scaler.step(optimizer)
        scaler.update()

        batch_size = len(batch["ids"])
        total_loss += loss.item() * batch_size
        total_cases += batch_size

        if step % 100 == 0:
            print(
                f"Epoch {epoch} | "
                f"Batch {step}/{len(train_loader)} | "
                f"Loss: {total_loss / total_cases:.4f}"
            )

    dev_probs, dev_targets = collect_probabilities(
        lsi_model, dev_loader
    )

    threshold, dev_f1 = choose_threshold(
        dev_probs, dev_targets
    )

    print(
        f"Epoch {epoch} | "
        f"Train loss: {total_loss / total_cases:.4f} | "
        f"Dev macro F1: {dev_f1:.4f} | "
        f"Threshold: {threshold:.2f}"
    )

    improved = dev_f1 > best_f1

    if improved:
        best_f1 = dev_f1
        bad_epochs = 0
    else:
        bad_epochs += 1

    state = {
        "model_state_dict": lsi_model.state_dict(),
        "model_settings": model_settings,
        "statute_ids": statute_ids,
        "threshold": threshold,
        "epoch": epoch,
        "dev_f1": dev_f1,
        "best_f1": best_f1,
        "bad_epochs": bad_epochs,
    }

    if improved:
        torch.save(state, OUTPUT_DIR / "best_f1.pt")
        print("Saved best model.")

    state.update({
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
    })

    torch.save(state, OUTPUT_DIR / "latest.pt")

    if bad_epochs >= PATIENCE:
        print("Early stopping.")
        break

In [ ]:
saved = torch.load(
    OUTPUT_DIR / "best_f1.pt",
    map_location="cpu",
    weights_only=True,
)

lsi_model.load_state_dict(saved["model_state_dict"])

BEST_THRESHOLD = float(saved["threshold"])
statute_ids = saved["statute_ids"]

del saved

dev_probs, dev_targets = collect_probabilities(
    lsi_model, dev_loader
)

dev_predictions = (
    dev_probs >= BEST_THRESHOLD
).astype(np.int32)

print("Best threshold:", BEST_THRESHOLD)

print(
    "Macro F1:",
    f1_score(
        dev_targets,
        dev_predictions,
        average="macro",
        zero_division=0,
    ),
)

print(
    "Micro F1:",
    f1_score(
        dev_targets,
        dev_predictions,
        average="micro",
        zero_division=0,
    ),
)

print(
    "Subset accuracy:",
    accuracy_score(dev_targets, dev_predictions),
)

In [ ]:
test_cases = read_cases(
    LSI_DIR / "Test/test.json",
    labeled=False,
)

test_dataset = StatuteDataset(test_cases)
test_loader = make_loader(test_dataset)


@torch.inference_mode()
def predict_statutes(model, loader, threshold):
    model.eval()
    results = []

    for batch in loader:
        batch = move_batch(batch)

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=use_amp,
        ):
            logits = get_logits(model, batch)

        predictions = (
            torch.sigmoid(logits) >= threshold
        ).cpu().numpy()

        for case_id, row in zip(
            batch["ids"], predictions
        ):
            labels = [
                int(statute_ids[index])
                for index in np.flatnonzero(row)
            ]

            results.append({
                "id": case_id,
                "pred": labels,
            })

    return results


test_predictions = predict_statutes(
    lsi_model,
    test_loader,
    BEST_THRESHOLD,
)

prediction_path = OUTPUT_DIR / "prediction.json"

with open(prediction_path, "w", encoding="utf-8") as f:
    json.dump(
        test_predictions,
        f,
        ensure_ascii=False,
        indent=2,
    )

with zipfile.ZipFile(
    OUTPUT_DIR / "submission.zip",
    "w",
    zipfile.ZIP_DEFLATED,
) as archive:
    archive.write(
        prediction_path,
        arcname="prediction.json",
    )

print("Saved:", OUTPUT_DIR / "submission.zip")